# Analiza streszczeń literatury młodzieżowej

## Import bibliotek

In [1]:
import json
import nltk
import string
import morfeusz2
import pandas as pd
import operator as op
import itertools as it
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

ModuleNotFoundError: No module named 'nltk'

In [ ]:
nltk.download('punkt_tab')
nltk.pathsec.ENFORCE = False

# Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()

# Wstępne przygotowanie dokumentów

In [ ]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

In [ ]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]  # Skip the first four lines of the file
stopwords_file.close()
stopwords

In [ ]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        perpl, lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None
    lemmas = (disambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))

In [ ]:
for file_name in documents_dict:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])
    documents_dict[file_name] = lemmatize(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish') if word not in stopwords])
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

In [ ]:
for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths['change'] = lengths['post'] - lengths['pre']
lengths['change_percentage'] = ((lengths['change'] / lengths['pre']) * 100).round(2)
lengths

# Utworzenie macierzy częstości

In [ ]:
documents_df = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])
documents_df

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df['content'])
matrix_tfidf = TfidfVectorizer().fit_transform(documents_df['content'])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100
sparsity_tfidf = (1 - (matrix_tfidf.nnz / (matrix_tfidf.shape[0] * matrix_tfidf.shape[1]))) * 100